# Lab 6 Instructor Demos

Four things `neo4j-agent-memory` does that `01_agent_memory.ipynb` has no room
for. This notebook is **run and read**, not an exercise. Nothing here is on the
participant path, and nothing below depends on anything above it except the setup.

| Demo | The question it answers | Minutes |
|---|---|---|
| 1. Correction | When a technician says "no, it was the *left* engine", what happens to what the agent already believed? | 6 |
| 2. Learned preferences | Can the agent remember *how* someone likes to be answered, not just what about? | 5 |
| 3. Shift handoff | Can it brief the incoming shift on what the outgoing shift was worried about? | 6 |
| 4. Routing memory | Can the agent remember which tool worked last time and skip the two that did not? | 8 |

Demo 4 is the one to run if there is only time for one. It is the one that
changes what people think an agent memory is *for*.

**Setup.** This notebook can run standalone, but it is much better after
`01_agent_memory.ipynb`, because Demo 3 reads the shift history that notebook
seeded. Run notebook 01 first if you can.

In [ ]:
%pip install /Volumes/databricks-neo4j-workshop/aircraft/raw_data/neo4j_agent_memory-0.5.1.dev0+mentions-py3-none-any.whl httpx>=0.27.0

In [ ]:
dbutils.library.restartPython()

In [ ]:
import sys

sys.path.insert(0, ".")

from memory import MemorySession, ensure_labs_on_path

ensure_labs_on_path()

from data_utils import read_neo4j_secrets, secret_scope_name

SECRET_SCOPE = secret_scope_name(spark)
NEO4J_DATABASE = read_neo4j_secrets(dbutils, SECRET_SCOPE)["database"]

session = MemorySession.open_from_secrets(
    dbutils, SECRET_SCOPE, database=NEO4J_DATABASE
)
client = session.client

print(f"Connected. Secret scope: {SECRET_SCOPE}")

In [ ]:
# Names every demo below uses. Defined once here so any single demo can be run
# on its own, which is the point of the table above.
from datetime import datetime, timezone

from neo4j_agent_memory.schema.models import EntityRef

from tools import SUPERVISOR_MODEL, get_llm

TECH = "tech:okafor"
N10004 = EntityRef(name="N10004", type="AIRCRAFT", label="Aircraft")
N10011 = EntityRef(name="N10011", type="AIRCRAFT", label="Aircraft")

llm = get_llm(SUPERVISOR_MODEL)
print(f"Supervisor model: {SUPERVISOR_MODEL}")

---

## Demo 1: A correction, and what happens to the old belief

**The problem.** An agent is told something. Later it is told something that
contradicts it. Most memory systems now hold both, retrieve whichever the vector
search likes better, and answer confidently either way. That is not a memory,
that is a coin flip with extra steps.

**What this library does instead.** A correction does not delete the old belief
and does not sit beside it. It **supersedes** it: an edge is written from old to
new, and the old one's `valid_until` is stamped with the moment it stopped being
true.

So the graph holds two answers to "what did we believe about this aircraft":
what we believe *now*, and what we believed *last Tuesday*. Both are queryable.
That second one is not an academic nicety. It is what you need when someone asks
why the agent said what it said in an incident review.

In [ ]:
# Monday. The technician tells the agent which engine.
original = session.run(
    client.long_term.add_preference(
        category="fault-location",
        preference="The EGT exceedance on N10004 is on the number two engine.",
        context="reported during the Monday night shift",
        user_identifier=TECH,
        applies_to=[N10004],
    )
)
print(f"Recorded: {original.preference}")
print(f"  id={original.id}")

In [ ]:
# Let a moment pass so the time-travel query below has two sides to it.
BEFORE_CORRECTION = datetime.now(timezone.utc)

import time

time.sleep(2)

In [ ]:
# Tuesday. The borescope says otherwise.
correction = session.run(
    client.long_term.add_preference(
        category="fault-location",
        preference="The EGT exceedance on N10004 is on the number ONE engine. "
                   "Borescope confirmed Tuesday; the Monday report was wrong.",
        context="corrected after borescope inspection",
        user_identifier=TECH,
        applies_to=[N10004],
    )
)

session.run(client.long_term.supersede_preference(original.id, correction.id))
print("Superseded.")

In [ ]:
# What does the agent believe now?
print("ACTIVE, as of right now:\n")
for pref in session.run(
    client.long_term.get_preferences_for(user_identifier=TECH, active_only=True)
):
    print(f"  {pref.preference}")

# What did it believe on Monday, before the correction landed?
print(f"\nAS OF {BEFORE_CORRECTION:%H:%M:%S}, before the correction:\n")
for pref in session.run(
    client.long_term.get_preferences_for(
        user_identifier=TECH, active_only=False, as_of=BEFORE_CORRECTION
    )
):
    print(f"  {pref.preference}")

In [ ]:
# The correction as it exists in the graph. One edge, one timestamp.
for row in session.cypher(
    """
    MATCH (old:Preference)-[:SUPERSEDED_BY]->(new:Preference)
    RETURN old.preference AS superseded,
           old.valid_until AS stopped_being_true,
           new.preference  AS replacement
    """
):
    print(f"  was:      {row['superseded'][:70]}")
    print(f"  until:    {row['stopped_being_true']}")
    print(f"  now:      {row['replacement'][:70]}\n")

**Point to make out loud.** Nothing was deleted. The wrong answer is still there,
still attached to the aircraft, still attached to the technician who gave it, and
stamped with the moment it stopped being true. An audit can reconstruct exactly
what the agent knew at any point. Delete the row instead and you have an agent
that cannot explain itself.

---

## Demo 2: Preferences the agent learns and keeps

A preference here is not a UI setting. It is anything about *how* this person
wants to work that the agent should stop being told twice.

Two kinds, and the difference is the demo:

- **Global.** "Always give me the part number, not the description." Applies to
  everything this technician asks.
- **Scoped.** "On N10011, ignore the EGT sensor, it reads five degrees high."
  Applies to one aircraft, and it is attached to that aircraft's node.

The scoped one is where the graph earns its keep. The preference hangs off the
same `N10004` node the maintenance events hang off, so "what do I need to know
before touching this aircraft" is one traversal from the aircraft, not a lookup
in a preferences table keyed by a string.

In [ ]:
session.run(
    client.long_term.add_preference(
        category="output-format",
        preference="Give me part numbers, not part descriptions. I order from the number.",
        user_identifier=TECH,
    )
)

session.run(
    client.long_term.add_preference(
        category="sensor-trust",
        preference="On N10011 the EGT sensor reads about 5 degrees high. "
                   "Subtract before comparing against limits.",
        context="known calibration offset, work order pending",
        user_identifier=TECH,
        applies_to=[N10011],
    )
)

print("Two preferences recorded: one global, one scoped to N10011.")

In [ ]:
print("Everything this technician has told the agent about how to work:\n")
for pref in session.run(
    client.long_term.get_preferences_for(user_identifier=TECH, active_only=True)
):
    scope = pref.context or "everything"
    print(f"  [{pref.category}] {pref.preference}")
    print(f"      applies: {scope}\n")

In [ ]:
# The traversal that matters: start at the aircraft, find what to know first.
for row in session.cypher(
    """
    MATCH (ac:Aircraft {tail_number: $tail})<-[:APPLIES_TO]-(p:Preference)
    WHERE p.valid_until IS NULL
    RETURN p.category AS category, p.preference AS preference
    """,
    {"tail": "N10011"},
):
    print(f"  [{row['category']}] {row['preference']}")

**Point to make out loud.** That last query starts at an aircraft, not at a user.
Any technician who touches `N10011` gets the calibration warning, because it is a
property of the aircraft's situation, not of one person's profile. A preferences
table keyed by user cannot express that without a join nobody writes.

---

## Demo 3: The shift handoff

The real job. Night shift ends, day shift starts, and the outgoing crew writes
three lines on a whiteboard that lose most of what they knew.

Notebook 01 seeded five technicians' conversations. The agent has been listening
to all of them. Ask it what the outgoing shift was worried about.

**This one only works if `01_agent_memory.ipynb` has been run.** If the counts
come back empty, run its Section 5.

In [ ]:
rows = session.cypher(
    "MATCH (:Message)-[r:MENTIONS]->(:Aircraft) RETURN count(r) AS mentions"
)
print(f"Seeded mentions found: {rows[0]['mentions']}")
if rows[0]["mentions"] == 0:
    print("Run Section 5 of 01_agent_memory.ipynb first, or Demo 3 has nothing to read.")

In [ ]:
# The handoff, straight out of the graph. No LLM involved yet.
HANDOFF = """
MATCH (u:User)-[:HAS_CONVERSATION]->(:Conversation)-[:HAS_MESSAGE]->(m:Message)
      -[:MENTIONS]->(ac:Aircraft)
WITH ac, collect(DISTINCT u.identifier) AS technicians, collect(m.content) AS said
OPTIONAL MATCH (ac)<-[:AFFECTS_AIRCRAFT]-(ev:MaintenanceEvent)
WITH ac, technicians, said,
     count(ev) AS events,
     count(CASE WHEN ev.severity = 'CRITICAL' THEN 1 END) AS critical
RETURN ac.tail_number AS aircraft, size(technicians) AS crew, technicians,
       events, critical, said
ORDER BY size(technicians) DESC, critical DESC
"""

briefing = session.cypher(HANDOFF)
for row in briefing:
    print(f"{row['aircraft']}  ({row['crew']} technician(s): {', '.join(row['technicians'])})")
    print(f"   maintenance record: {row['events']} events, {row['critical']} critical")
    for line in row["said"]:
        print(f"   - {line}")
    print()

In [ ]:
# Now hand that to the model and ask for the whiteboard version.
facts = "\n".join(
    f"{r['aircraft']}: asked about by {r['crew']} technician(s) "
    f"({', '.join(r['technicians'])}); maintenance record shows {r['events']} events, "
    f"{r['critical']} critical. They said: " + " | ".join(r["said"])
    for r in briefing
)

prompt = (
    "You are writing the shift handover note for an aircraft maintenance team.\n"
    "Below is what the outgoing shift asked about, and what the maintenance\n"
    "record says about the same aircraft.\n\n"
    "Lead with any aircraft where the two disagree: several people worried about\n"
    "an aircraft the record calls healthy, or nobody asking about one that is\n"
    "failing. Be brief. No preamble.\n\n"
    f"{facts}\n"
)

print(llm.invoke(prompt).content)

**Point to make out loud.** The interesting line in that note is always the
disagreement. The record and the conversation are two independent readings of the
same fleet, and where they diverge is where a supervisor should look. You cannot
compute that divergence without both, in one place, joined on the same node.

---

## Demo 4: The agent remembers which tool worked

This is the one.

The Lab 5 supervisor picks a tool by asking a language model, every single time,
from scratch. Ask it the same question twice and it reasons from zero twice. Ask
it a question it got wrong last week and it makes the same mistake.

A **reasoning trace** records what the agent tried, which tool it called, which
entities that call touched, and whether it worked. Next time a similar question
arrives, the agent can look up what happened last time before it decides.

That turns routing from a fixed prompt into something that improves with use.

In [ ]:
from neo4j_agent_memory.core.memory import ToolCallStatus

# Record a run that went badly: the wrong tool, then the right one.
trace = session.run(
    client.reasoning.start_trace(
        "demo-routing",
        "What is the EGT trend on N10011 over the last month?",
        user_identifier=TECH,
    )
)

step1 = session.run(
    client.reasoning.add_step(
        trace.id,
        thought="Sounds like a graph question. Try Cypher.",
        action="cypher_node",
        observation="Neo4j has no time series. Returned aircraft metadata only.",
    )
)
session.run(
    client.reasoning.record_tool_call(
        step1.id,
        "cypher_node",
        {"question": "EGT trend on N10011"},
        result="no sensor readings in the graph",
        status=ToolCallStatus.ERROR,
        touched_entities=[N10011],
    )
)

step2 = session.run(
    client.reasoning.add_step(
        trace.id,
        thought="Sensor time series lives in the lakehouse, not the graph. Use Genie.",
        action="genie_node",
        observation="Returned 30 days of EGT readings with a downward margin trend.",
    )
)
session.run(
    client.reasoning.record_tool_call(
        step2.id,
        "genie_node",
        {"question": "EGT readings for N10011, last 30 days"},
        result="30 rows",
        status=ToolCallStatus.SUCCESS,
        touched_entities=[N10011],
    )
)

session.run(
    client.reasoning.complete_trace(
        trace.id,
        outcome="genie_node answers sensor-trend questions. cypher_node cannot: "
                "the graph holds no time series.",
        success=True,
    )
)
print(f"Trace recorded: {trace.id}")

In [ ]:
# A new question arrives. Different aircraft, different wording, same shape.
NEW_QUESTION = "Show me the vibration trend for N10004 over the past few weeks."

similar = session.run(
    client.reasoning.get_similar_traces(NEW_QUESTION, limit=3, success_only=True)
)

print(f"New question: {NEW_QUESTION}\n")
if not similar:
    print("No similar trace found yet.")
for past in similar:
    print(f"  Similar past task: {past.task}")
    print(f"  What we learned:   {past.outcome}\n")

In [ ]:
# Feed that to the supervisor prompt and let it route with hindsight.
hindsight = "\n".join(f"- {t.task} -> {t.outcome}" for t in similar) or "(nothing similar)"

prompt = (
    "You are routing a question to one of three tools:\n"
    "  genie_node    - sensor time series in the lakehouse\n"
    "  cypher_node   - fleet structure and maintenance history in Neo4j\n"
    "  graphrag_node - maintenance manuals\n\n"
    "Here is what happened on similar questions before:\n"
    f"{hindsight}\n\n"
    f"Question: {NEW_QUESTION}\n\n"
    "Answer with the tool name and one sentence of why."
)

print(llm.invoke(prompt).content)

In [ ]:
# The audit trail. Which agent decisions touched this aircraft, and did they work?
#
# Note where :TOUCHED hangs from. It is written on the ReasoningStep, not the
# ToolCall, so the traversal is step -> entity for the audit and step -> call
# for the mechanics.
for row in session.cypher(
    """
    MATCH (:ReasoningTrace)-[:HAS_STEP]->(s:ReasoningStep)-[:USES_TOOL]->(tc:ToolCall)
    MATCH (s)-[:TOUCHED]->(ac:Aircraft)
    RETURN ac.tail_number AS aircraft, tc.tool_name AS tool,
           tc.status AS status, s.observation AS observation
    ORDER BY tc.tool_name
    """
):
    print(f"  {row['aircraft']}  {row['tool']:<14} {row['status']}")
    print(f"      {row['observation']}\n")

That `MATCH (s)-[:TOUCHED]->(ac:Aircraft)` only resolves because notebook 01
adopted the fleet's `Aircraft` nodes. Without adoption the library would have
created its own `N10011` `Entity` and the pattern would return nothing, which is
the same lesson as Section 6 of notebook 01 arriving from a different direction.

In [ ]:
# The library also keeps a running score per tool, updated on every call.
# This is what a routing policy could read instead of a prompt.
for row in session.cypher(
    """
    MATCH (t:Tool)
    RETURN t.name AS tool, t.total_calls AS calls,
           t.successful_calls AS succeeded, t.failed_calls AS failed
    ORDER BY t.total_calls DESC
    """
):
    print(f"  {row['tool']:<14} {row['calls']} call(s), "
          f"{row['succeeded']} succeeded, {row['failed']} failed")

**Point to make out loud.** Two things just happened that are worth separating.

**The routing got better.** The agent did not reason about Genie versus Cypher
from first principles. It looked up what happened last time a question of this
shape came through, and that lookup cost one vector search instead of one LLM
call.

**The audit trail is a graph.** The last query starts at an aircraft and walks
back through tool calls to reasoning steps to the trace. "Which agent decisions
touched this aircraft, and which of them failed" is a traversal. In a system
where traces are log lines and the fleet is a database, it is a support ticket.

That is the whole argument for this architecture, and it is why the memory lives
in the graph rather than beside it.

---

## Cleanup

The demos above wrote preferences and a reasoning trace into the graph. Removing
them leaves the fleet data and the seeded shift history alone.

Skip this if you want to poke at what the demos built.

In [ ]:
CLEANUP = [
    "MATCH (p:Preference) DETACH DELETE p",
    "MATCH (tc:ToolCall) DETACH DELETE tc",
    "MATCH (s:ReasoningStep) DETACH DELETE s",
    "MATCH (t:ReasoningTrace) DETACH DELETE t",
    "MATCH (t:Tool) DETACH DELETE t",
]

# session.cypher() is read-only by design, so writes go through the driver.
from tools import open_driver_from_secrets

driver = open_driver_from_secrets(dbutils, SECRET_SCOPE)
for statement in CLEANUP:
    summary = driver.execute_query(statement, database_=NEO4J_DATABASE)[2]
    print(f"  {statement.split()[1]:<22} deleted {summary.counters.nodes_deleted}")
driver.close()

session.close()
print("\nClosed.")